# 20.14 — Compilation & runtimes

ML compilers turn model graphs into faster executable plans by fusing operations, specializing shapes, and targeting hardware kernels.

This gap topic is authored from the Part 20 plan and lesson content. The notebook simulates graph fusion, constant folding, shape specialization, warmup, numerical tolerance, and production p95 latency using CPU-only NumPy.

Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2017)


def percentile(values, q):
    return float(np.percentile(np.asarray(values, dtype=float), q))


def softmax(logits, temperature=1.0):
    scaled = np.asarray(logits, dtype=float) / temperature
    shifted = scaled - np.max(scaled, axis=-1, keepdims=True)
    exp_values = np.exp(shifted)
    return exp_values / np.sum(exp_values, axis=-1, keepdims=True)


def cross_entropy(target_probs, pred_probs):
    clipped = np.clip(pred_probs, 1e-9, 1.0)
    return float(-np.sum(target_probs * np.log(clipped)))


def expected_calibration_error(confidence, correct, bins=10):
    confidence = np.asarray(confidence, dtype=float)
    correct = np.asarray(correct, dtype=float)
    edges = np.linspace(0.0, 1.0, bins + 1)
    total = len(confidence)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (confidence >= lo) & (confidence < hi)
        if hi == 1.0:
            mask = (confidence >= lo) & (confidence <= hi)
        if np.any(mask):
            gap = abs(float(np.mean(confidence[mask])) - float(np.mean(correct[mask])))
            ece += float(np.mean(mask)) * gap
    return ece


def make_f17_workload_ladder(seed=2017):
    local_rng = np.random.default_rng(seed)
    return [
        {
            "rung": "D1",
            "name": "tiny hand-check",
            "requests": 60,
            "precision": "int8",
            "scale": 1.8 / 127.0,
            "shape": (8,),
            "load": 1.0,
            "data": np.array([0.73, -0.40, 1.20, -1.70, 0.05, 0.88, -0.91, 1.55]),
        },
        {
            "rung": "D2",
            "name": "clean tensor workload",
            "requests": 400,
            "precision": "int8",
            "scale": 2.5 / 127.0,
            "shape": (64, 32),
            "load": 1.7,
            "data": local_rng.normal(0.0, 0.65, size=(64, 32)),
        },
        {
            "rung": "D3",
            "name": "outliers and bad calibration",
            "requests": 900,
            "precision": "int8",
            "scale": 1.0 / 127.0,
            "shape": (128, 48),
            "load": 2.4,
            "data": np.vstack([
                local_rng.normal(0.0, 0.45, size=(124, 48)),
                local_rng.normal(0.0, 2.2, size=(4, 48)),
            ]),
        },
        {
            "rung": "D4",
            "name": "digits-like classifier weights",
            "requests": 1600,
            "precision": "mixed int8/fp16",
            "scale": 3.0 / 127.0,
            "shape": (10, 64),
            "load": 3.5,
            "data": local_rng.normal(0.0, 0.85, size=(10, 64)),
        },
        {
            "rung": "D5",
            "name": "production-scale load simulation",
            "requests": 5000,
            "precision": "per-channel int8",
            "scale": 4.0 / 127.0,
            "shape": (256, 128),
            "load": 6.0,
            "data": local_rng.normal(0.0, 0.95, size=(256, 128)),
        },
    ]


def preview_ladder(ladder):
    for rung in ladder:
        data = rung.get("data")
        shape = getattr(data, "shape", rung.get("shape"))
        print(f"{rung['rung']} | {rung['name']} | shape={shape} | requests={rung['requests']} | precision={rung['precision']}")
        print("sample:", np.asarray(data).reshape(-1)[:5])


## The concept, built once (D1)

Use $\text{speedup}=T_{eager}/T_{compiled}$ and $|y_c-y_e|\le\epsilon$. The lesson's fused kernel changes $1.8$ ms to $1.1$ ms and two $4$ MB reads to one $4$ MB read.

In [ ]:

def compile_plan(ops, static_shape):
    eager_latency_ms = sum(op["latency_ms"] for op in ops)
    compiled_latency_ms = sum(op["latency_ms"] for op in ops if not op.get("folded", False))
    fusion_saving_ms = 0.0
    traffic_saving_mb = 0.0
    if any(op.get("fusible", False) for op in ops):
        fusion_saving_ms = 0.7
        traffic_saving_mb = 4.0
    compiled_latency_ms = max(compiled_latency_ms - fusion_saving_ms, 0.05)
    elements = int(np.prod(static_shape))
    return {
        "eager_latency_ms": eager_latency_ms,
        "compiled_latency_ms": compiled_latency_ms,
        "speedup": eager_latency_ms / compiled_latency_ms,
        "traffic_saving_mb": traffic_saving_mb,
        "elements": elements,
    }

lesson_ops = [
    {"name": "read_a", "latency_ms": 0.9, "fusible": True},
    {"name": "read_b", "latency_ms": 0.9, "fusible": True},
]
lesson_plan = compile_plan(lesson_ops, (32, 128))
constant_fold = 3 * 4 + 2
kernel_speedup = 1.8 / 1.1
kernel_saving = 1.8 - 1.1
numeric_diff = abs(0.731 - 0.730)

print("traffic saving", lesson_plan["traffic_saving_mb"])
print("constant", constant_fold)
print("elements", lesson_plan["elements"])
print("kernel speedup", round(kernel_speedup, 3))
print("kernel saving", round(kernel_saving, 1))
print("numeric diff", round(numeric_diff, 3))

assert lesson_plan["traffic_saving_mb"] == 4.0
assert constant_fold == 14
assert lesson_plan["elements"] == 4096
assert round(kernel_speedup, 3) == 1.636
assert round(kernel_saving, 1) == 0.7
assert numeric_diff <= 0.005


The runtime evaluator separates compile warmup from steady-state latency. It also records shape mismatches and tolerance failures as operational outputs, not notebook side effects.

In [ ]:

def make_compilation_ladder(seed=2017):
    local_rng = np.random.default_rng(seed + 14)
    specs = [
        ("D1", "tiny three-op graph", (1, 8), 20, 2.0, 0.2),
        ("D2", "clean operator chain", (16, 64), 300, 7.0, 0.6),
        ("D3", "dynamic shapes and warmup", (None, 128), 800, 25.0, 1.3),
        ("D4", "request-shape trace", (32, 128), 1600, 45.0, 2.4),
        ("D5", "production serving simulation", (64, 256), 5000, 120.0, 4.8),
    ]
    ladder = []
    for rung, name, shape, requests, compile_ms, eager_base in specs:
        observed_batches = local_rng.choice([16, 32, 48, 64, 96], size=requests)
        data = local_rng.normal(eager_base, 0.15 * eager_base + 0.05, size=requests)
        ladder.append({
            "rung": rung,
            "name": name,
            "shape": shape,
            "requests": requests,
            "compile_ms": compile_ms,
            "eager_base_ms": eager_base,
            "observed_batches": observed_batches,
            "precision": "compiled fp32",
            "load": requests / 1000,
            "data": data,
        })
    return ladder


def evaluate_compilation_workload(rung, include_warmup=False):
    eager = np.asarray(rung["data"], dtype=float)
    compiled = eager * 0.62 + 0.08
    shape = rung["shape"]
    if shape[0] is not None:
        mismatch = np.mean(rung["observed_batches"] != shape[0])
        compiled = compiled + mismatch * 0.35
    else:
        mismatch = 0.0
        compiled = compiled + 0.25
    if include_warmup:
        compiled = compiled.copy()
        warmup_count = max(1, int(0.10 * len(compiled)))
        compiled[:warmup_count] = compiled[:warmup_count] + rung["compile_ms"]
    speedup = percentile(eager, 95) / percentile(compiled, 95)
    y_eager = np.array([0.730, 0.120, -0.040])
    y_compiled = y_eager + np.array([0.001, -0.001, 0.0005])
    tolerance_ok = bool(np.max(np.abs(y_compiled - y_eager)) <= 0.005)
    return {
        "p95_eager_ms": percentile(eager, 95),
        "p95_compiled_ms": percentile(compiled, 95),
        "speedup": speedup,
        "shape_mismatch": float(mismatch),
        "tolerance_ok": tolerance_ok,
        "artifact": compiled[:80],
    }


## The dataset ladder

D1 is a tiny graph, D2 is a clean operator chain, D3 adds dynamic shapes and warmup, D4 uses a request-shape trace, and D5 simulates production compiled/eager serving.

In [ ]:

ladder = make_compilation_ladder()
preview_ladder(ladder)


## Run the SAME method across D1–D5

In [ ]:

compile_results = []
for rung in ladder:
    result = evaluate_compilation_workload(rung)
    result["rung"] = rung["rung"]
    result["name"] = rung["name"]
    compile_results.append(result)

print("rung | p95_eager | p95_compiled | speedup | shape_mismatch | tolerance")
for result in compile_results:
    print(f"{result['rung']} | {result['p95_eager_ms']:.2f} | {result['p95_compiled_ms']:.2f} | {result['speedup']:.2f}x | {result['shape_mismatch']:.2f} | {result['tolerance_ok']}")


## Results visualization

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for ax, result in zip(axes[0], compile_results):
    ax.plot(result["artifact"], linewidth=1.0)
    ax.set_title(result["rung"] + " compiled latencies")
    ax.set_ylabel("ms")

metric_curve = [result["speedup"] for result in compile_results]
axes[1, 0].plot([rung["rung"] for rung in ladder], metric_curve, marker="o")
axes[1, 0].set_title("speedup curve")
for ax, result in zip(axes[1, 1:], compile_results[1:]):
    ax.bar(["eager", "compiled", "mismatch"], [result["p95_eager_ms"], result["p95_compiled_ms"], result["shape_mismatch"]])
    ax.set_title(result["rung"] + " operational panel")
plt.tight_layout()


## Pitfall on D5: benchmarking without warmup

Compilation time is not steady-state inference time. Reproduce p95 latency with compile time mixed in, then separate warmup from serving measurements.

In [ ]:

d5 = ladder[-1]
wrong = evaluate_compilation_workload(d5, include_warmup=True)
fixed = evaluate_compilation_workload(d5, include_warmup=False)

print("wrong p95", round(wrong["p95_compiled_ms"], 2))
print("fixed p95", round(fixed["p95_compiled_ms"], 2))
print("fixed speedup", round(fixed["speedup"], 2))
assert fixed["p95_compiled_ms"] < wrong["p95_compiled_ms"]
assert fixed["tolerance_ok"]


## Evaluate it + Practice

- Metric: track steady-state speedup; compare with the no-skill baseline `eager graph with no fusion`.
- Cheap sanity check: constant folding should produce 14.
- Ablation: include compile warmup in the measured request stream.
- Failure signals: shape mismatches or numerical tolerance failures.
- Production guardrail: monitor the metric by rung instead of averaging D1 with D5.

Practice 1: Change one D5 load or precision setting and predict the metric before running.

Practice 2: Add one operational constraint, such as memory budget or p95 latency budget, and mark the first rung that violates it.

Practice 3: Write a one-line launch rule that uses the metric and one pitfall guardrail.